# Task 3 — Kafka Topic Design

## Approach and reasoning

Four topics carry the four event categories required by the lab:

| Topic | Message key | Partitions | `cleanup.policy` | Rationale |
|---|---|---|---|---|
| `cpg.nodes` | node id | 6 | compact | high volume, parallel sink consumption |
| `cpg.edges` | edge id | 6 | compact | high volume |
| `cpg.metadata` | file id | 1 | compact | one message per file, ordering trivial |
| `cpg.errors` | file id | 1 | delete, 7 days | diagnostics, not state |

### Why the key is the stable element id
Keying by the element's stable id gives us three properties at once:

1. **Ordering** — all versions of one element hash to the same partition, so a
   later version can never be overtaken by an earlier one.
2. **Compaction safety** — with `cleanup.policy=compact`, Kafka retains only the
   latest value per key. The topic therefore tracks the *current* graph rather
   than the cumulative event history, and its size stays bounded across replays.
3. **Alignment with the sink** — compaction semantics ("latest per key wins")
   mirror the `MERGE` semantics used in Neo4j, so the two layers agree.

### Message envelope
Every message carries `schema_version` (forward compatibility, required by the
lab) and `event_time` as ISO-8601 UTC (required by the lab), plus `file_id` and
`file_hash` so downstream consumers can perform generation-based cleanup.

The four contracts are formalised as JSON Schema in `schemas/`.

In [ ]:
import os, json
os.chdir("..")
for name in ["node", "edge", "metadata", "error"]:
    s = json.load(open(f"schemas/{name}_event.schema.json"))
    print(f"{name:9} required: {s['required']}")

### Validate a real produced event against its schema

In [ ]:
import json, jsonschema
schema = json.load(open("schemas/node_event.schema.json"))
event = json.loads(open("reports/events/samples/cpg.nodes.sample.jsonl").readline())["value"]
jsonschema.validate(event, schema)
print("node event VALID against schema")
print("schema_version:", event["schema_version"], "| event_time:", event["event_time"])

### Create the topics on the live cluster

> **Run this cell against the running Docker stack.** It requires
> `docker compose up -d` to have completed and the broker healthcheck to pass.

In [ ]:
!python src/kafka_setup/create_topics.py --bootstrap localhost:9092

### Confirm the topics exist and inspect their configuration

In [ ]:
!docker exec broker kafka-topics --bootstrap-server localhost:9092 --list
!docker exec broker kafka-topics --bootstrap-server localhost:9092 --describe --topic cpg.nodes

### Sample live messages from the topic (after the parser has run)

In [ ]:
!docker exec broker kafka-console-consumer --bootstrap-server localhost:9092 \
    --topic cpg.nodes --from-beginning --max-messages 2 --timeout-ms 10000

## Reflection

*(Replace this with what actually happened on your machine.)*

**What worked.** Log compaction was the non-obvious win. Because keys are stable
ids, a replayed file overwrites its own messages instead of appending new ones,
so topic size tracks the current graph rather than growing with every replay.

**What to watch.** Partition count is a one-way door in Kafka — increasing it
later re-hashes keys and breaks the per-element ordering guarantee. We picked 6
for the high-volume topics up front rather than starting at 1.